# Notebook 39 — Advanced Planning

Strategic reasoning: break complex goals into trees, search for optimal actions,
let an agent grow its own task queue, and summarise arbitrarily large corpora.

| Component | Role |
|---|---|
| `HierarchicalDecomposer` | Recursively decompose a goal into a task tree |
| `MCTSPlanner` | Monte Carlo Tree Search — UCB1 action selection |
| `AutoGPTQueue` | Agent-driven self-directed task execution loop |
| `HierarchicalSummariser` | Recursive summarise-of-summaries over large corpora |

## Setup

In [ ]:
import sys
sys.path.insert(0, '../sdk')

from multigen.planning_advanced import (
    TaskNode,
    HierarchicalDecomposer,
    MCTSNode,
    MCTSPlanner,
    TaskQueueItem,
    AutoGPTResult,
    AutoGPTQueue,
    SummaryNode,
    HierarchicalSummariser,
)

print('All imports OK')

---
## Section 1 — Hierarchical Task Decomposition

`HierarchicalDecomposer` takes a high-level goal string and breaks it into a
tree of sub-tasks by repeatedly calling a `decompose_fn`.  The function
returns a list of sub-task strings (empty list = leaf node).

After decomposition an optional `execute_fn` is called on each leaf, with
results aggregated bottom-up.

In this example the decompose function uses a fixed knowledge base to simulate
an LLM breaking down the goal "Build a blog platform".

In [ ]:
import asyncio

# Knowledge base: known sub-task breakdowns keyed by task name
SUBTASK_MAP = {
    'Build a blog platform': [
        'Design system architecture',
        'Implement backend API',
        'Build frontend UI',
        'Set up infrastructure',
    ],
    'Design system architecture': [
        'Define data models',
        'Choose technology stack',
    ],
    'Implement backend API': [
        'User authentication service',
        'Post management endpoints',
        'Comment service',
    ],
    'Build frontend UI': [
        'Home page and navigation',
        'Post editor component',
        'Reader view component',
    ],
    'Set up infrastructure': [
        'Configure CI/CD pipeline',
        'Set up cloud hosting',
    ],
    # Depth-2 nodes are leaves (not in map) — returns []
}

def mock_decompose(task: str, depth: int, ctx: dict) -> list:
    """Mock LLM decomposition — returns sub-tasks from the knowledge base."""
    return SUBTASK_MAP.get(task, [])   # [] means this task is atomic

def mock_execute(task: str, ctx: dict) -> str:
    """Mock executor — in production this would call an LLM or tool."""
    return f'[DONE] {task}'

decomposer = HierarchicalDecomposer(
    decompose_fn=mock_decompose,
    execute_fn=mock_execute,
    max_depth=3,
    max_children=8,
)

async def run_decomposition():
    root = await decomposer.decompose('Build a blog platform')
    return root

root = asyncio.run(run_decomposition())
print(f'Root: {root.task}')
print(f'Status: {root.status}')
print()

In [ ]:
# Pretty-print the full task tree
def print_tree(node: TaskNode, indent: int = 0) -> None:
    prefix = '    ' * indent
    icon = '🌿' if node.is_leaf() else '📁'
    status_tag = f'[{node.status}]' if node.status != 'pending' else ''
    result_preview = ''
    if node.result and isinstance(node.result, str):
        result_preview = f' => "{node.result}"'
    print(f'{prefix}{icon} {node.task} {status_tag}{result_preview}')
    for child in node.children:
        print_tree(child, indent + 1)

print('Task tree for "Build a blog platform":')
print_tree(root)
print()

leaves = root.all_leaves()
print(f'Total leaf tasks: {len(leaves)}')
print(f'Max depth reached: {max(n.depth for n in leaves)}')

In [ ]:
# Inspect the JSON-serialisable dict representation
import json
tree_dict = root.to_dict()
# Truncate the print for readability
def summarise_tree(d, depth=0):
    indent = '  ' * depth
    children_info = f'  ({len(d["children"])} children)' if d['children'] else ''
    print(f'{indent}- {d["task"]} [{d["status"]}]{children_info}')
    for child in d['children']:
        summarise_tree(child, depth + 1)

print('Tree dict structure:')
summarise_tree(tree_dict)

---
## Section 2 — MCTS Planning

`MCTSPlanner` uses the Upper Confidence Bound 1 (UCB1) algorithm to balance
exploration vs exploitation:

```
UCB1(node) = V(node)/N(node)  +  C × sqrt(ln(N(parent)) / N(node))
```

Where `V` = total value, `N` = visit count, `C` = exploration constant (√2).

This example models a 5×5 grid world.  The agent starts at `(0,0)` and tries
to reach the goal at `(4,4)`.  The `simulate_fn` rewards states closer to the
goal, so MCTS quickly learns to prefer moves toward `(4,4)`.

In [ ]:
import math, random

GRID_SIZE = 5
GOAL = (4, 4)

def actions_fn(state):
    """Legal moves from the current grid position."""
    r, c = state['pos']
    moves = []
    if r > 0: moves.append('up')
    if r < GRID_SIZE - 1: moves.append('down')
    if c > 0: moves.append('left')
    if c < GRID_SIZE - 1: moves.append('right')
    return moves

def transition_fn(state, action):
    """Apply an action and return the new state."""
    r, c = state['pos']
    if action == 'up':    r -= 1
    elif action == 'down':  r += 1
    elif action == 'left':  c -= 1
    elif action == 'right': c += 1
    return {'pos': (r, c), 'step': state.get('step', 0) + 1}

def simulate_fn(state):
    """Rollout reward: higher when close to the goal (normalised 0–1)."""
    r, c = state['pos']
    max_dist = math.sqrt(2) * (GRID_SIZE - 1)
    dist = math.sqrt((r - GOAL[0])**2 + (c - GOAL[1])**2)
    # Add small random noise to simulate stochastic rollout
    noise = random.uniform(-0.05, 0.05)
    return max(0.0, min(1.0, 1.0 - dist / max_dist + noise))

def is_terminal_fn(state):
    """Terminal when the agent reaches the goal."""
    return state['pos'] == GOAL

planner = MCTSPlanner(
    actions_fn=actions_fn,
    transition_fn=transition_fn,
    simulate_fn=simulate_fn,
    is_terminal_fn=is_terminal_fn,
    n_simulations=200,
    exploration=1.414,
)

async def run_mcts():
    start = {'pos': (0, 0), 'step': 0}
    best_action = await planner.plan(start)
    return best_action

best_action = asyncio.run(run_mcts())
print(f'Starting position: (0, 0)  |  Goal: {GOAL}')
print(f'Best first action selected by MCTS: "{best_action}"')
print('(Expected: "right" or "down" — toward the goal)')

In [ ]:
# Trace a full path from start to goal using MCTS at each step
async def trace_path(max_steps=20):
    state = {'pos': (0, 0), 'step': 0}
    path = [state['pos']]

    for step in range(max_steps):
        if is_terminal_fn(state):
            break
        action = await planner.plan(state)
        state = transition_fn(state, action)
        path.append(state['pos'])

    return path

path = asyncio.run(trace_path())
print(f'Path traced by MCTS ({len(path)} steps):')
for i, pos in enumerate(path):
    marker = ' <- GOAL' if pos == GOAL else ''
    print(f'  Step {i:2d}: {pos}{marker}')

# Visualise the path on the grid
grid = [['.' for _ in range(GRID_SIZE)] for _ in range(GRID_SIZE)]
for i, (r, c) in enumerate(path):
    if (r, c) == (0, 0): grid[r][c] = 'S'
    elif (r, c) == GOAL: grid[r][c] = 'G'
    else: grid[r][c] = str(i % 10)

print('\nGrid visualisation (S=start, G=goal, numbers=path steps):')
for row in grid:
    print('  ' + ' '.join(row))

---
## Section 3 — AutoGPT-Style Task Queue

`AutoGPTQueue` runs an agent in a self-directed loop.  On each step the agent
receives a context dict containing `goal`, `current_task`, `completed_tasks`,
and `step`.  It returns a dict that may contain:

- `"new_tasks": [str, ...]` — tasks to enqueue for future execution
- `"done": True` — signals the agent is finished
- `"output": <any>` — result value for this task
- `"error": str` — non-fatal error (task marked failed)

This simulates an AutoGPT-style agent planning and executing "Build a news
aggregator" step by step.

In [ ]:
# Task expansion rules — simulates an LLM deciding what to do next
TASK_PLAN = {
    'Build a news aggregator': {
        'new_tasks': [
            'Identify top 5 news RSS feeds',
            'Build RSS parser module',
            'Implement de-duplication logic',
            'Build article ranking algorithm',
        ],
        'output': 'Goal decomposed into 4 sub-tasks',
    },
    'Identify top 5 news RSS feeds': {
        'output': 'BBC, Reuters, AP, Guardian, NYT RSS feeds identified',
        'new_tasks': [],
    },
    'Build RSS parser module': {
        'new_tasks': ['Write unit tests for RSS parser'],
        'output': 'RSS parser implemented using feedparser',
    },
    'Write unit tests for RSS parser': {
        'output': '12 unit tests written, all passing',
    },
    'Implement de-duplication logic': {
        'output': 'URL + title hashing deduplication implemented',
    },
    'Build article ranking algorithm': {
        'new_tasks': ['Evaluate ranking on test corpus'],
        'output': 'TF-IDF + recency ranking implemented',
    },
    'Evaluate ranking on test corpus': {
        'output': 'Ranking evaluated: NDCG=0.87 on 200-article test set',
        'done': True,
    },
}

step_log = []

def news_aggregator_agent(ctx: dict) -> dict:
    task = ctx['current_task']
    step = ctx['step']
    plan = TASK_PLAN.get(task, {'output': f'Completed: {task}'})
    step_log.append({
        'step': step,
        'task': task,
        'new_tasks': plan.get('new_tasks', []),
        'done': plan.get('done', False),
    })
    return plan

queue = AutoGPTQueue(
    agent_fn=news_aggregator_agent,
    max_steps=15,
    max_tasks=50,
)

async def run_autogpt():
    result = await queue.run(
        goal='Build a news aggregator',
        initial_ctx={'project': 'news-agg-v1'},
    )
    return result

autogpt_result = asyncio.run(run_autogpt())

print('=== AutoGPT Execution Log ===')
for entry in step_log:
    new_t = f" + enqueued: {entry['new_tasks']}" if entry['new_tasks'] else ''
    done  = ' [DONE]' if entry['done'] else ''
    print(f"  Step {entry['step']:2d}: {entry['task']}{new_t}{done}")

print()
print('=== AutoGPTResult ===')
print(f'  goal:            {autogpt_result.goal}')
print(f'  steps_taken:     {autogpt_result.steps_taken}')
print(f'  completed tasks: {len(autogpt_result.completed_tasks)}')
print(f'  failed tasks:    {autogpt_result.failed_tasks}')
print(f'  success:         {autogpt_result.success}')
print(f'  final output:    {autogpt_result.output}')

In [ ]:
# Show all completed tasks in order
print('Completed task sequence:')
for i, task in enumerate(autogpt_result.completed_tasks, 1):
    print(f'  {i:2d}. {task}')

---
## Section 4 — Hierarchical Summarisation

`HierarchicalSummariser` handles large document collections that exceed the
context window of any single LLM call.  It:

1. Splits chunks into groups of `chunk_size`
2. Summarises each group concurrently
3. Recurses on the summaries
4. Returns a `SummaryNode` tree rooted at the final summary

This scales to arbitrarily large corpora — a 1000-chunk document with
`chunk_size=5` needs only `ceil(log_5(1000)) = 4` levels.

In [ ]:
# Generate 20 fake article paragraphs about climate change
chunks = [
    f'[Chunk {i+1:02d}] {topic}'
    for i, topic in enumerate([
        'Global temperatures have risen 1.1°C above pre-industrial levels.',
        'Arctic sea ice extent reached a record low in September 2023.',
        'Permafrost thaw releases stored methane accelerating warming.',
        'Ocean heat content hit a record high for the 7th consecutive year.',
        'CO2 concentrations surpassed 420 ppm at Mauna Loa observatory.',
        'Extreme heat events are now 5× more frequent than in the 1950s.',
        'Sea level rise is accelerating due to ice sheet melt in Greenland.',
        'Coral reef bleaching events have doubled in frequency since 1980.',
        'Renewable energy now accounts for 30% of global electricity.',
        'EV adoption grew 35% in 2023, displacing oil demand.',
        'Carbon capture technology costs fell 40% over the past decade.',
        'Deforestation in the Amazon slowed by 50% under new policy.',
        'Wildfire season length has extended by 3 months globally.',
        'Flooding events caused $300bn in economic losses in 2023.',
        'Drought conditions affected 40% of global cropland in 2023.',
        'The IPCC projects 2°C could be crossed by 2040 under high emissions.',
        'Climate migration displaced an estimated 24 million people in 2023.',
        'Methane emissions from agriculture remain the hardest to reduce.',
        'Green hydrogen production capacity tripled over the last 5 years.',
        'The Paris Agreement review mechanism showed insufficient pledges.',
    ])
]

print(f'Number of input chunks: {len(chunks)}')
print('First 3 chunks:')
for c in chunks[:3]:
    print(f'  {c}')

In [ ]:
summarise_call_count = {'n': 0}

def mock_summarise(chunk_list: list) -> str:
    """Mock summariser: extracts key phrases from each chunk."""
    summarise_call_count['n'] += 1
    # Extract the core fact from each chunk (strip prefix)
    facts = []
    for chunk in chunk_list:
        # Strip '[Chunk NN] ' prefix if present, keep first 60 chars
        text = chunk.split('] ', 1)[-1] if '] ' in chunk else chunk
        facts.append(text[:55])
    joined = '; '.join(facts[:3])  # show up to 3 facts in summary
    ellipsis = '...' if len(facts) > 3 else ''
    return f'Summary(n={len(chunk_list)}): {joined}{ellipsis}'

summariser = HierarchicalSummariser(
    summarise_fn=mock_summarise,
    chunk_size=5,      # summarise 5 chunks at a time
    max_levels=8,
)

async def run_summarisation():
    root_node = await summariser.summarise(chunks)
    return root_node

root_summary = asyncio.run(run_summarisation())

print(f'Summarise function called {summarise_call_count["n"]} times')
print()
print('Root summary:')
print(f'  {root_summary.summary}')

In [ ]:
# Visualise the summary tree structure
def print_summary_tree(node: SummaryNode, indent: int = 0) -> None:
    prefix = '  ' * indent
    summary_preview = (node.summary or '')[:70]
    if len(node.summary or '') > 70:
        summary_preview += '...'
    if node.children:
        print(f'{prefix}[Level {node.level}] {len(node.chunks)} chunks, {len(node.children)} sub-groups')
        print(f'{prefix}  Summary: "{summary_preview}"')
        for child in node.children:
            print_summary_tree(child, indent + 1)
    else:
        print(f'{prefix}[Level {node.level}] {len(node.chunks)} chunks (leaf)')
        print(f'{prefix}  Summary: "{summary_preview}"')

print('Hierarchical Summary Tree:')
print_summary_tree(root_summary)
print()
print(f'Tree structure (to_dict):')
def tree_shape(d, depth=0):
    indent = '  ' * depth
    print(f'{indent}Level {d["level"]}: n_chunks={d["n_chunks"]}, children={len(d["children"])}')
    for child in d['children']:
        tree_shape(child, depth + 1)

tree_shape(root_summary.to_dict())

---
## Summary

```
HierarchicalDecomposer  →  break goals into task trees; execute leaves in parallel
MCTSPlanner             →  UCB1 tree search; optimal for discrete action spaces
AutoGPTQueue            →  agent drives its own task queue; bounded by max_steps
HierarchicalSummariser  →  O(log N) LLM calls for arbitrarily large documents
```

**Choosing the right planner:**

| Scenario | Best component |
|---|---|
| Known decomposition rules / static DAG | `HierarchicalDecomposer` |
| Uncertain environment, discrete actions | `MCTSPlanner` |
| Open-ended goal, agent decides sub-tasks | `AutoGPTQueue` |
| Summarise corpus > context window | `HierarchicalSummariser` |

**Scaling note:** `HierarchicalDecomposer` and `HierarchicalSummariser`
use `asyncio.gather` for sibling nodes, so the wall-clock time scales as
`O(depth)` rather than `O(total nodes)` when using async LLM backends.